# Volatility Band Width: Estimating and Backtesting Volatility Bands for the S&P 500
### K. Mohamed Harith
**Erdős Institute -- Quantitative Finance Bootcamp, Fall 2026** · [GitHub repository](https://github.com/Harry140896/Volatility-band-width)

**Question.** Can volatility models give reliable 95% bands for the S&P 500's next-day return? And if so, how should volatility be estimated to make them more reliable?

This notebook builds 95% bands from four volatility models: constant-volatility GBM, rolling-window GBM, the Exponentially Weighted Moving Average (EWMA) and GARCH(1,1) (Generalised Autoregressive Conditional Heteroskedasticity, with one lag each of squared returns and of variance). It has three parts:
1. **Building the bands:** the four models, each calibrated on 2005–2017.
2. **Backtesting:** recording when each model's band is missed on 2018–2026.
3. **Testing the misses:** Kupiec's test (are there 5% misses?) and Christoffersen's test (are the misses independent?), followed by conclusions.

**About the data.** We use daily closing prices of the S&P 500 index (Yahoo Finance ticker ^GSPC) from January 2005 to the 3rd November 2026, downloaded with `yfinance` (adjusted for splits and dividends where relevant) and saved to `data/gspc.csv` so that every run uses the same data. We work with daily log returns, $r_t = \ln(P_t / P_{t-1})$.

The data is split chronologically:
- **Calibration (fit) period: 2005–2017**, about 3,270 trading days, including the 2008 financial crisis;
- **Test period: 2018–present**, about 2,220 trading days, including the February 2018 volatility spike, the COVID-19 crash of March 2020, the 2022 bear market and the April 2025 sell-off.

Both periods contain a major crisis, so the models learn from one crisis and are tested on others; both also contain calm years. All model settings are chosen on the fit period only. The S&P 500 is a price index and excludes *dividends*; these are spread across many companies and days, so they barely affect daily volatility.

The following code cell contains all the required imports for running this notebook.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm,chi2
import seaborn as sns
import matplotlib.pyplot as plt
import yfinance as yf
from pathlib import Path
from arch import arch_model
from scipy.special import xlogy  #For Kupiec Test

import warnings
warnings.filterwarnings("ignore", message=".*inplace keyword.*")
print(np.__version__)
print(yf.__version__)
print(pd.__version__)
#print(arch.__version__)

2.5.4
1.7.0
3.1.0


In [2]:
#Setting the seed for reproducible results

SEED = 44
rng = np.random.default_rng(SEED)


# The folders set up
DATA_DIR = Path("data")
CSV_PATH = DATA_DIR/ "gspc.csv"


## Data downloading
df  = yf.Ticker("^GSPC").history(start = '2005-01-01', auto_adjust = True)
df.index = df.index.tz_localize(None)
df.to_csv(CSV_PATH)
print(f'Downloaded and saved the CSV file in {CSV_PATH}')



## Loading the Close Price for the S&P500 Data
close = df['Close']  

# Log Returns Series
log_returns = np.log(close).diff().dropna()




fit_log_returns=log_returns['2005':'2017']
test_log_returns = log_returns['2018':]


log_returns.groupby(log_returns.index.year).std()*np.sqrt(252)

Downloaded and saved the CSV file in data\gspc.csv


Date
2005    0.102694
2006    0.100165
2007    0.160214
2008    0.410199
2009    0.272660
2010    0.180617
2011    0.233543
2012    0.127581
2013    0.110709
2014    0.113824
2015    0.155092
2016    0.131121
2017    0.066874
2018    0.170988
2019    0.124895
2020    0.346871
2021    0.131068
2022    0.241912
2023    0.130825
2024    0.126663
2025    0.186469
2026    0.130101
Name: Close, dtype: float64

## 1. Motivation: constant volatility and its band

### The GBM model
Under Geometric Brownian Motion, the log return of a stock over a short interval $\Delta t$ (in years) is normally distributed:
$$
r_t = \ln\frac{S_t}{S_{t-\Delta t}} = \left(\mu - \tfrac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t}\; X, \qquad X \sim N(0,1),
$$
where $\mu$ is the drift and $\sigma$ the **annual volatility**, assumed **constant**.

### From the model to a 95% band
For one trading day, $\Delta t = 1/252$. Write $m = (\mu - \sigma^2/2)\,\Delta t$ for the daily drift and $s = \sigma\sqrt{\Delta t}$ for the **daily volatility**, so that $r = m + s\,X$. Since $P(-1.96 \le X \le 1.96) = 0.95$,
$$
P\big(m - 1.96\,s \;\le\; r \;\le\; m + 1.96\,s\big) = 0.95 .
$$
Over one day the drift is negligible compared with the volatility (for the S&P 500, $m \approx 0.0003$ against $s \approx 0.012$), so we set $m = 0$. The 95% band for tomorrow's log return is then
$$
\big[-1.96\,s,\; +1.96\,s\big].
$$
Throughout this notebook, a **volatility band** means this 95% prediction interval for the next day's log return: its width is set entirely by the volatility estimate. Equivalently, tomorrow's price lies in $[S_{t-1}e^{-1.96s},\, S_{t-1}e^{+1.96s}]$ with 95% probability.

### Estimating the volatility from data
The volatility $\sigma$ cannot be observed directly; it has to be estimated from past prices. Given daily log returns $r_1, \dots, r_n$, the sample standard deviation
$$
s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n} (r_i - \bar r)^2}
$$
estimates the **daily** volatility $\sigma\sqrt{\Delta t}$, and the annual volatility is $\sigma = s/\sqrt{\Delta t} = s\sqrt{252}$ (252 trading days a year). Using the whole fit period 2005–2017 gives a single daily volatility, and therefore one fixed band for every day of the test period.

**The question this section asks:** is one fixed band good enough?